# 2-6. LLM-as-a-Judge와 심화 평가

⏱ **소요시간**: 30분  
🧩 **심화/선택**  
<!-- optional -->

이전 노트북(2-5)에서 RAGAS 4대 지표를 훑었다면, 여기서는 **판사로서의 LLM(LLM-as-a-judge)** 방법론을 직접 구현하고 RAGAS의 심화 지표까지 확장한다. 운영 단계에서 '사람 평가는 비싸고 BLEU는 거짓말한다'는 현실을 다루는 핵심 세션이다.

## 학습 목표
- LLM-as-a-judge의 세 가지 유형(**single-grading · pairwise · reference-based**)을 직접 구현하고 상황별 장단점을 설명한다.
- RAGAS의 심화 지표(**answer correctness · answer similarity · context entity recall · noise sensitivity**)를 적용할 수 있다.
- judge의 체계적 편향(**position / verbosity / self-preference / familiar-style**)을 인지하고 **순서 교차·앙상블·캘리브레이션**으로 완화할 수 있다.

## 핵심 키워드
`LLM-as-a-judge` `pairwise comparison` `G-Eval` `RAGAS advanced` `position bias` `calibration` `judge ensemble`

> 🔒 **폐쇄망 주의**: judge도 로컬 모델을 쓰게 되면 평가 신뢰도가 하락한다. `LLM_PROVIDER=ollama`인 경우 최소 `qwen2.5:14b-instruct` 이상 권장.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

## 1. 평가의 스펙트럼

생성형 LLM 출력물을 정량화하는 방법은 수십 년간 진화해왔다. 각 방법은 **신뢰도 vs 비용 vs 재현성**의 트레이드오프 위에 놓인다.

| 방식 | 대표 지표 | 비용/속도 | 상관관계(사람 판단과) | 주 용도 |
|---|---|---|---|---|
| **n-gram 기반** | BLEU, ROUGE, METEOR | 극저 / 즉시 | 낮음 (표면 일치) | 기계번역 회귀테스트 |
| **임베딩 기반** | BERTScore, MoverScore | 저 / 빠름 | 중간 (의미 유사도) | 요약 품질 |
| **모델 기반(작은)** | COMET, BLEURT | 중 / 보통 | 중상 | MT 학술 평가 |
| **LLM-as-a-judge** | G-Eval, MT-Bench, RAGAS | 중상 / 느림 | 높음 (사람 0.8+) | 개방형 QA·RAG |
| **사람 평가** | Likert, pairwise | 매우 높음 / 매우 느림 | 기준점 (1.0) | 릴리즈 게이트 |

**핵심 통찰**: BLEU/ROUGE는 표현을 달리한 정답을 틀렸다고 판단한다. 예: 정답이 "14일 이내"일 때 모델이 "2주 내"라 답하면 BLEU≈0이지만 실제로는 정답이다. 이 문제를 풀기 위해 등장한 것이 LLM judge이다.

In [ ]:
# BLEU의 한계 직접 관찰 — 같은 의미, 다른 표현
from difflib import SequenceMatcher

reference = '이용자는 14일 이내에 서면으로 철회할 수 있다.'
candidate_a = '이용자는 14일 이내에 서면으로 철회할 수 있다.'        # 완전 일치
candidate_b = '고객은 2주 이내에 문서로 취소 가능하다.'               # 같은 의미, 다른 단어
candidate_c = '이용자는 14일 이내에 서면으로 철회할 수 없다.'        # 한 글자("없다") 차이 — 반대 의미

def surface_sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

for label, cand in [('A 동일', candidate_a), ('B 패러프레이즈', candidate_b), ('C 반대의미', candidate_c)]:
    print(f'{label:15s} | surface_sim = {surface_sim(reference, cand):.2f} | "{cand}"')

B(패러프레이즈)는 의미상 **정답**인데도 표면 유사도가 낮고, C(반대의미)는 의미상 **오답**인데도 표면 유사도가 매우 높다. 이런 실패가 쌓이면 지표는 더 이상 품질 신호로 쓸 수 없다.

## 2. LLM-as-a-Judge 원리

**Zheng et al. 2023 (MT-Bench / Chatbot Arena)**: GPT-4 judge가 사람 평가자와 약 **80% 이상** 일치한다는 것을 보였다. 사람끼리의 평가자 간 일치율과 비슷한 수준이다. 이로 인해 LLM judge는 사실상 업계 표준이 됐다.

### 왜 작동하는가
- **일관성**: 같은 rubric을 모든 샘플에 동일하게 적용 (사람은 피로·분위기로 흔들림).
- **확장성**: 수천 건 평가를 분 단위로 처리.
- **근거 제시**: rubric 기반이면 점수와 함께 설명을 요구할 수 있다.

### 편향 카탈로그 (반드시 인지할 것)
| 편향 | 정의 | 완화책 |
|---|---|---|
| **Position bias** | A/B 비교 시 첫 번째 또는 마지막을 선호 | 순서 교차(swap) 후 집계 |
| **Verbosity bias** | 긴 답변을 우수하다 판단 | 길이 제약을 rubric에 명시, 길이 정규화 |
| **Self-preference** | judge가 자기 스타일과 닮은 답변 선호 | 서로 다른 모델 앙상블 |
| **Familiar-style** | 영어식·markdown 답변을 과대평가 | 도메인별 rubric 보강 |
| **Sycophancy** | 프롬프트의 힌트(예: "좋은 답변")에 동조 | 중립적 프롬프트, 근거 먼저 요구 |

아래 셀에서 **position bias를 실측**해본다. 동일한 두 답변 A/B를 순서만 바꿔 judge에게 물었을 때, 순서에 관계없이 선호 비율이 50:50이 되는 것이 이상적이지만 실제로는 한쪽이 유의미하게 선호된다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 동일 질문에 대한 두 답변 — 의도적으로 품질이 비슷하도록 설계
question = '전자금융거래 이용계약은 체결 후 얼마 동안 철회할 수 있나요?'
answer_x = '이용자는 계약 체결일로부터 14일 이내에 서면으로 철회할 수 있습니다. 다만 이미 서비스가 제공된 경우 철회가 제한됩니다.'
answer_y = '계약일로부터 14일 이내에 서면 철회가 가능하며, 이미 서비스 이용이 시작된 경우에는 철회가 불가합니다.'

judge = get_llm(temperature=0)

pairwise_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 금융 QA 품질 평가자입니다. 두 답변 중 더 나은 쪽을 선택하세요. 답은 반드시 "A", "B", 또는 "TIE" 한 단어만 출력하세요.'),
    ('human', '질문: {q}\n\n[A] {a}\n\n[B] {b}\n\n더 나은 쪽:'),
])
chain = pairwise_prompt | judge | StrOutputParser()

N_TRIALS = 6  # 비용 고려 소규모 — 실전에서는 30+ 권장
orig_votes, swap_votes = [], []
for i in range(N_TRIALS):
    orig_votes.append(chain.invoke({'q': question, 'a': answer_x, 'b': answer_y}).strip().upper()[:1])
    swap_votes.append(chain.invoke({'q': question, 'a': answer_y, 'b': answer_x}).strip().upper()[:1])

print(f'원래 순서(X=A, Y=B) 선택률: A={orig_votes.count("A")}/{N_TRIALS}  B={orig_votes.count("B")}/{N_TRIALS}  T={orig_votes.count("T")}/{N_TRIALS}')
print(f'뒤바꾼 순서(Y=A, X=B) 선택률: A={swap_votes.count("A")}/{N_TRIALS}  B={swap_votes.count("B")}/{N_TRIALS}  T={swap_votes.count("T")}/{N_TRIALS}')
print('\n→ 만약 judge가 실제로 X와 Y를 동등하게 본다면, A 선택률이 두 번 모두 비슷해야 한다.')
print('→ 순서에 따라 A 선택률이 크게 달라지면 position bias가 존재한다는 증거.')

## 3. 구현 1 — Single-answer grading (Pydantic 구조화 출력)

가장 단순한 형태. rubric 하나를 주고 1~5점 점수와 근거를 받는다. LangChain의 **Structured Output**을 써서 파싱 오류를 원천 차단한다.

In [ ]:
import sys; sys.path.insert(0, '../..')
import json
from pathlib import Path
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from common import get_llm

baseline_path = Path('./_capstone_baseline.json')
if not baseline_path.exists():
    raise FileNotFoundError('먼저 04_rag_pipeline_lcel.ipynb를 실행해 _capstone_baseline.json 을 생성하세요.')
baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
print(f'샘플 {len(baseline)}개 로드 — 첫 질문: {baseline[0]["question"][:40]}...')

class SingleScore(BaseModel):
    """단일 답변 채점 스키마."""
    score: int = Field(description='1~5점, 1=매우 나쁨, 5=매우 좋음', ge=1, le=5)
    reason: str = Field(description='채점 근거를 2~3문장으로 한국어로 작성')

RUBRIC = """다음 기준으로 1~5점을 매기세요.
1점: 질문과 무관하거나 사실 오류.
2점: 부분적으로 관련 있으나 중요한 정보가 누락/틀림.
3점: 대체로 맞지만 모호하거나 근거 부족.
4점: 정확하고 명확하며 근거가 충분.
5점: 정확·명확하며 법적 근거 조항이나 예외 사항까지 포함.
길이가 길다고 점수를 올리지 마세요(verbosity bias 주의)."""

single_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 금융 약관 QA 품질 평가자입니다.\n\n{rubric}\n\n채점 전 먼저 관련 사실을 속으로 정리한 뒤 점수를 결정하세요(chain-of-thought 허용).'),
    ('human', '질문:\n{question}\n\n참고 컨텍스트:\n{contexts}\n\n평가 대상 답변:\n{answer}'),
])

judge = get_llm(temperature=0).with_structured_output(SingleScore)
single_chain = single_prompt | judge

single_results = []
for row in baseline:
    ctx = '\n---\n'.join(row['contexts'][:3])
    out = single_chain.invoke({'rubric': RUBRIC, 'question': row['question'], 'contexts': ctx, 'answer': row['answer']})
    single_results.append({'question': row['question'][:40] + '...', 'score': out.score, 'reason': out.reason})

import pandas as pd
pd.DataFrame(single_results)

## 4. 구현 2 — Pairwise comparison (position-bias 완화 포함)

MT-Bench·Chatbot Arena가 쓰는 방식. 두 답변 중 승자를 고른다. **같은 페어를 순서를 바꿔 두 번** 평가하면 position bias의 영향을 상쇄할 수 있다.

**집계 규칙(간단형)**:
- 두 번 모두 A 승 → A 승
- 두 번 모두 B 승 → B 승
- 서로 다르거나 TIE 포함 → TIE

기준이 되는 S2-4 baseline 답변(temperature=0)과, 같은 질문을 **temperature=0.7**로 재생성한 답변을 맞붙여본다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal
from common import get_llm

# 1) 대조군 답변 생성 — 동일 질문을 temperature=0.7 로 재생성
generator = get_llm(temperature=0.7)

gen_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 금융 약관 QA 보조자입니다. 제공된 컨텍스트만 근거로 간결히 답하세요.'),
    ('human', '질문: {question}\n\n컨텍스트:\n{contexts}\n\n답변:'),
])
gen_chain = gen_prompt | generator

variant_answers = []
for row in baseline:
    ctx = '\n---\n'.join(row['contexts'][:3])
    variant = gen_chain.invoke({'question': row['question'], 'contexts': ctx}).content
    variant_answers.append(variant)

print('재생성 답변 샘플(첫 번째):')
print(variant_answers[0][:200], '...')

In [ ]:
# 2) Pairwise judge — 순서 교차(swap) 평가
class Pairwise(BaseModel):
    winner: Literal['A', 'B', 'TIE'] = Field(description='더 나은 답변 또는 TIE')
    reason: str = Field(description='한국어 2~3문장')

pair_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 금융 QA 품질 평가자입니다. 두 답변 중 더 정확하고 근거가 명확한 쪽을 고르세요.\n'
     '**길이가 긴 답변을 우수하다고 판단하지 마세요(verbosity bias).**\n'
     '판단이 어려우면 TIE를 허용합니다.'),
    ('human', '질문:\n{question}\n\n컨텍스트:\n{contexts}\n\n[A]\n{a}\n\n[B]\n{b}'),
])
pair_judge = get_llm(temperature=0).with_structured_output(Pairwise)
pair_chain = pair_prompt | pair_judge

def aggregate(orig: str, swap: str) -> str:
    """원래 순서에서의 승자(orig)와 뒤바꾼 순서에서의 승자(swap)를 합의한다."""
    # swap에서 'A'는 원래 B였으므로, swap을 원래 좌표로 변환
    swap_mapped = {'A': 'B', 'B': 'A', 'TIE': 'TIE'}[swap]
    if orig == swap_mapped and orig != 'TIE':
        return orig
    return 'TIE'

pairwise_rows = []
for row, variant in zip(baseline, variant_answers):
    ctx = '\n---\n'.join(row['contexts'][:3])
    A, B = row['answer'], variant  # A = baseline, B = temperature=0.7 variant
    orig = pair_chain.invoke({'question': row['question'], 'contexts': ctx, 'a': A, 'b': B})
    swap = pair_chain.invoke({'question': row['question'], 'contexts': ctx, 'a': B, 'b': A})
    final = aggregate(orig.winner, swap.winner)
    pairwise_rows.append({
        'question': row['question'][:35] + '...',
        'orig_order': orig.winner,
        'swap_order': swap.winner,
        'final': final,
    })

import pandas as pd
pair_df = pd.DataFrame(pairwise_rows)
print(pair_df)
print('\n최종 집계 (A=baseline, B=temperature=0.7 variant):')
print(pair_df['final'].value_counts().to_dict())

> **해석 힌트**: `orig_order`와 `swap_order`가 서로 다르면(예: A→B, swap에서도 A(=원래 B)를 골랐다면 둘 다 원래 좌표계로 B) 그 샘플은 **판단이 불안정**하다는 뜻이다. 실전에서는 그런 샘플만 사람이 들여다본다.

## 5. 구현 3 — Reference-based (G-Eval 스타일)

**G-Eval (Liu et al. 2023)**: judge에게 먼저 **평가 단계를 스스로 생성**시킨 뒤 그 단계에 따라 점수를 매기게 하는 기법. 여기서는 그 축약판으로, reference 답변을 주고 **coherence / relevance / consistency**를 각각 1~5점으로 채점한다.

- **coherence**: 답변이 논리적으로 흐르는가
- **relevance**: 질문에 대해 핵심 정보를 다루는가
- **consistency**: reference와 사실이 일치하는가

In [ ]:
import sys; sys.path.insert(0, '../..')
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from common import get_llm

# 노트북 내 수작업 ground-truth (05 노트북과 동일한 정답 재활용)
GROUND_TRUTH = [
    '14일 이내에 서면으로 철회할 수 있으나, 이미 서비스가 제공되었거나 영업일 기준 3일 이내 증정이 끝난 경우 제한된다.',
    '이용자는 즉시 고객센터 또는 가장 가까운 영업점에 신고해야 하며, 선의무과실이 없다면 신고 이전 사고도 보상책임이 금융회사에 있다.',
    '수수료 인상 적용일 30일 전에 공지해야 한다.',
    '거래일로부터 1년 이내에 이의신청 가능하고, 금융회사는 접수일로부터 15영업일 이내 결과를 통지한다.',
    '원칙적으로 24시간 이용 가능하나, 시스템 점검·장애·운영상 필요에 따라 일시 제한될 수 있으며 사전 공지된다.',
]

class GEvalScore(BaseModel):
    coherence: int = Field(ge=1, le=5, description='논리적 흐름 1~5')
    relevance: int = Field(ge=1, le=5, description='질문 핵심 반영도 1~5')
    consistency: int = Field(ge=1, le=5, description='reference와의 사실 일치도 1~5')
    steps: list[str] = Field(description='채점 과정에서 스스로 생성한 2~4개의 확인 단계(한국어)')

geval_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 G-Eval 스타일 판사입니다. 다음 지시를 따르세요:\n'
     '1) 먼저 이 답변을 채점하기 위해 수행해야 할 2~4단계의 점검 절차를 한국어로 생성하세요(예: "수치 일치 확인", "예외 조항 포함 여부" 등).\n'
     '2) 그 절차에 따라 coherence·relevance·consistency를 1~5점으로 매기세요.\n'
     '각 축은 독립적으로 채점합니다.'),
    ('human',
     '질문: {question}\n\n'
     'Reference(정답):\n{reference}\n\n'
     '평가 대상 답변:\n{answer}'),
])
geval_chain = geval_prompt | get_llm(temperature=0).with_structured_output(GEvalScore)

geval_rows = []
for row, gt in zip(baseline, GROUND_TRUTH):
    out = geval_chain.invoke({'question': row['question'], 'reference': gt, 'answer': row['answer']})
    geval_rows.append({
        'question': row['question'][:30] + '...',
        'coherence': out.coherence,
        'relevance': out.relevance,
        'consistency': out.consistency,
        'n_steps': len(out.steps),
    })

import pandas as pd
g_df = pd.DataFrame(geval_rows)
print(g_df)
print('\n축별 평균:', g_df[['coherence','relevance','consistency']].mean().round(2).to_dict())

## 6. RAGAS 심화 지표

2-5 노트북에서 본 4개 지표 외에, RAGAS는 ground-truth가 있을 때 쓸 수 있는 **정답 기반 지표**를 추가로 제공한다.

| 지표 | 측정 대상 | 언제 쓰나 |
|---|---|---|
| **AnswerCorrectness** | 답변 vs ground_truth | 종합 정확도 (사실성 + 의미 유사도 가중합) |
| **AnswerSimilarity** | 답변 ↔ ground_truth 임베딩 | 표현은 달라도 의미가 일치하는지 |
| **ContextEntityRecall** | ground_truth 내 엔터티가 컨텍스트에 존재하는지 | 특정 기관명·수치 등 핵심 키워드를 retriever가 놓쳤는지 |
| **NoiseSensitivity** | 무관한 컨텍스트가 섞였을 때 답변이 흔들리는지 | 잡음 견딤성 (Context-Stuffing 내구성) |

In [ ]:
import sys; sys.path.insert(0, '../..')
from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
    context_entity_recall,
)
from common import get_llm, get_embeddings

ds = Dataset.from_list([
    {
        'question': r['question'],
        'answer': r['answer'],
        'contexts': r['contexts'],
        'ground_truth': gt,
    }
    for r, gt in zip(baseline, GROUND_TRUTH)
])

# common.get_llm / get_embeddings 를 주입 → OpenAI/Ollama/Local 모두 투명하게 동작
judge_llm = LangchainLLMWrapper(get_llm(temperature=0))
judge_emb = LangchainEmbeddingsWrapper(get_embeddings())

advanced_result = evaluate(
    ds,
    metrics=[answer_correctness, answer_similarity, context_entity_recall],
    llm=judge_llm,
    embeddings=judge_emb,
)
print(advanced_result)

In [ ]:
# 샘플별 테이블
import pandas as pd
adv_df = advanced_result.to_pandas()
cols = [c for c in ['question', 'answer_correctness', 'answer_similarity', 'context_entity_recall'] if c in adv_df.columns]
adv_df[cols].round(3)

In [ ]:
# NoiseSensitivity: 관련 없는 컨텍스트를 일부러 섞어 retriever의 잡음 견딤성을 본다.
# 운영에서는 실제 retriever의 k를 늘리면서 이 지표가 어떻게 움직이는지 추적한다.
from ragas.metrics import NoiseSensitivity

noise_metric = NoiseSensitivity()

# 잡음 섞기 — 각 샘플 컨텍스트 끝에 다른 샘플의 첫 번째 컨텍스트를 덧붙여 '노이즈' 를 삽입
noisy_rows = []
for i, r in enumerate(baseline):
    other_i = (i + 1) % len(baseline)
    noisy_ctx = list(r['contexts']) + [baseline[other_i]['contexts'][0]]
    noisy_rows.append({
        'question': r['question'],
        'answer': r['answer'],
        'contexts': noisy_ctx,
        'ground_truth': GROUND_TRUTH[i],
        'reference': GROUND_TRUTH[i],  # noise_sensitivity 는 reference 필드를 쓴다
    })

noise_ds = Dataset.from_list(noisy_rows)
noise_result = evaluate(noise_ds, metrics=[noise_metric], llm=judge_llm, embeddings=judge_emb)
print('NoiseSensitivity (낮을수록 잡음에 강함):', noise_result)

## 7. Calibration & 합의 — judge 앙상블과 Human-in-the-loop

단일 judge 한 번의 점수는 노이즈가 크다. 두 가지 완화책:

### (a) 같은 judge 반복 평가 (self-consistency)
temperature>0로 N번 샘플링하고 다수결/평균. 비용은 N배.

In [ ]:
import sys; sys.path.insert(0, '../..')
from collections import Counter
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from common import get_llm

class QuickScore(BaseModel):
    score: int = Field(ge=1, le=5)

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 금융 QA 평가자입니다. 아래 답변에 1~5점을 매기세요.'),
    ('human', '질문:{q}\n답변:{a}'),
])

# 같은 judge를 temperature=0.5로 세 번 호출 → 평균과 다수결
ensemble_judge = get_llm(temperature=0.5).with_structured_output(QuickScore)
chain = prompt | ensemble_judge

target = baseline[0]
scores = [chain.invoke({'q': target['question'], 'a': target['answer']}).score for _ in range(3)]
print(f'3회 점수: {scores}  평균: {sum(scores)/len(scores):.2f}  다수결: {Counter(scores).most_common(1)[0][0]}')

### (b) Human-in-the-loop 캘리브레이션 파이프라인

판사가 바르게 채점하고 있는지 **사람이 일부 샘플을 정답으로 라벨링**해 judge의 F1/정확도를 측정한다.

```
  ┌────────────────┐     ┌───────────────────┐
  │ 전체 평가셋 N   │ ──▶ │ LLM judge 채점     │
  └────────────────┘     └─────────┬─────────┘
           │ 20% 무작위              ▼
           ▼               ┌───────────────────┐
  ┌────────────────┐       │ 사람 채점 (gold)   │
  │ human 샘플 0.2N │ ────▶│                   │
  └────────────────┘       └─────────┬─────────┘
                                     ▼
                         judge vs human 일치율·F1
                         →  임계치 미달이면 rubric 재작성
```

**의사코드**:
```python
import random

human_sample_idx = random.sample(range(N), k=int(0.2 * N))
agree = 0
for i in human_sample_idx:
    judge_label = 'PASS' if judge_scores[i] >= 4 else 'FAIL'
    human_label = human_labels[i]                 # 수작업 라벨
    agree += int(judge_label == human_label)

agreement = agree / len(human_sample_idx)
# agreement < 0.75 → rubric 수정, few-shot 예시 추가, 더 큰 judge 모델로 교체
```

실전에서는 `scikit-learn`의 `cohen_kappa_score`로 **우연 일치 이상의 합의도**를 측정한다.

## 8. 실용 가이드 — 어떤 평가를 언제 쓸까

### 결정 트리
```
Q: ground-truth가 있는가?
├─ YES → RAGAS (answer_correctness, context_recall) + 필요시 G-Eval reference-based
└─ NO
   ├─ 두 시스템 비교? → Pairwise LLM-as-judge (순서 교차 필수)
   └─ 단일 시스템 모니터링?
      ├─ 환각 우려 → RAGAS faithfulness
      └─ 일반 품질 → Single-grading judge + 주기적 human sample
```

### 운영 체크리스트
- [ ] **비용**: judge 호출이 production 모델 호출의 몇 %를 차지하는가? (→ 샘플링 비율 조정)
- [ ] **속도**: CI에 넣을 거면 100 샘플 < 5분을 목표로 async 평가 (ragas `run_config`).
- [ ] **재현성**: judge 모델/프롬프트/버전을 평가 결과와 함께 기록. 모델 업데이트 시 점수 드리프트 발생.
- [ ] **감사(audit)**: 금융/의료라면 judge 프롬프트·근거까지 6개월 이상 보존 가능해야 한다.
- [ ] **편향 점검**: 분기별로 position-bias / verbosity 실측을 다시 돌려 judge 신뢰도 재확인.

### 폐쇄망 운영
- judge는 **가능한 한 큰 로컬 모델** 사용: `qwen2.5:14b-instruct` 이상 권장. `llama3.1:8b`는 rubric 따르기가 약하다.
- **RAGAS와 judge를 서로 다른 로컬 모델로** 돌려 self-preference 편향을 낮춘다 (예: 생성 = qwen2.5:7b, judge = qwen2.5:14b).
- 로컬 judge의 F1이 사람 대비 0.7 미만이면 **결정용이 아니라 모니터링용**으로만 쓴다.

## 정리

- **세 가지 judge 유형**: single-grading(빠름), pairwise(시스템 비교), reference-based/G-Eval(정답이 있을 때).
- judge는 본질적으로 **편향된 평가자**다. position·verbosity·self-preference 중 최소 하나는 반드시 실측하고 완화책을 붙여라.
- **RAGAS 심화 지표**(answer_correctness / entity_recall / noise_sensitivity)는 RAG 특유의 회귀를 잡아낸다.
- 최종 신뢰도는 **"judge + 20% human 샘플링 + 주기적 캘리브레이션"** 의 조합에서 나온다.
